# Radiant | NIH ChestX-ray14 transfer learning
This notebook uses the NIH Clinical Center ChestX-ray14 release attached from Kaggle. The Kaggle dataset is a mirror of the NIH release; the source, label limitations, patient-level split, exclusions, and measurements are recorded in the generated reports.

It keeps the official test patients separate, builds the validation split from development patients, uses only Pneumonia-positive and No Finding comparison images, and excludes other findings from this binary task. Labels are report-derived and are not clinical diagnoses.

The output candidate is eligible for serving only if it clears the validation promotion gate. Do not use the app for patient care.

In [ ]:
from pathlib import Path
import os, sys
import pandas as pd

input_root = Path('/kaggle/input')
metadata_files = sorted(input_root.rglob('Data_Entry_2017*.csv'))
if not metadata_files:
    raise FileNotFoundError('Attach the NIH Chest X-rays dataset input first.')
metadata_file = next((p for p in metadata_files if 'Data_Entry_2017_v2020' in p.name), metadata_files[0])
RAW = metadata_file.parent
assert (RAW / 'test_list.txt').is_file(), f'Missing official test list beside {metadata_file}'
assert (RAW / 'train_val_list.txt').is_file(), f'Missing official development list beside {metadata_file}'
print('Dataset root:', RAW)
print('Metadata file:', metadata_file.name)
print('Attached image files:', sum(1 for _ in RAW.rglob('*.png')))

In [ ]:
!pip -q install ImageHash==4.3.2
Path('/kaggle/working/app').mkdir(parents=True, exist_ok=True)
Path('/kaggle/working/artifacts').mkdir(parents=True, exist_ok=True)

In [ ]:
%%writefile /kaggle/working/app/__init__.py



In [ ]:
%%writefile /kaggle/working/app/model.py
"""ONNX-only serving; legacy checkpoints are evaluated offline."""
import hashlib
import json
import os
from pathlib import Path

BASE_DIR = Path(__file__).resolve().parent.parent
MODEL_PATH = Path(os.getenv("MODEL_PATH", str(BASE_DIR / "models/pneumonia_model.onnx")))
_MODEL = None
_METADATA = None

class ModelUnavailable(RuntimeError):
    pass

def get_model():
    global _MODEL, _METADATA
    if _MODEL is not None:
        return _MODEL
    if not MODEL_PATH.is_file() or not MODEL_PATH.with_suffix(".json").is_file():
        raise ModelUnavailable("Validated model artifact unavailable")
    try:
        metadata = json.loads(MODEL_PATH.with_suffix(".json").read_text(encoding="utf-8"))
        if metadata.get("validated") is not True or metadata.get("classes") != ["No Finding", "Pneumonia"]:
            raise ValueError("Unvalidated model or incompatible classes")
        if hashlib.sha256(MODEL_PATH.read_bytes()).hexdigest() != metadata.get("sha256"):
            raise ValueError("Checksum mismatch")
        if not 0.05 <= float(metadata["temperature"]) <= 20 or not metadata.get("version"):
            raise ValueError("Invalid metadata")
        import onnxruntime as ort
        options = ort.SessionOptions()
        options.intra_op_num_threads = 1
        options.inter_op_num_threads = 1
        session = ort.InferenceSession(str(MODEL_PATH), options, providers=["CPUExecutionProvider"])
        _MODEL, _METADATA = session, metadata
        return session
    except Exception as exc:
        raise ModelUnavailable("Model validation failed") from exc

def model_metadata():
    get_model()
    return dict(_METADATA)


In [ ]:
%%writefile /kaggle/working/app/utils.py
LANGUAGE_LABEL_TO_CODE = {
    "english": "en",
    "hindi": "hi",
    "marathi": "mr",
    "en": "en",
    "hi": "hi",
    "mr": "mr",
}

SUPPORTED_LANGS = {"en", "hi", "mr"}
DISCLAIMER = "Consult a doctor for medical advice."
CONFIDENCE_THRESHOLD = 0.8


def normalize_language(language: str | None) -> str:
    if not language:
        return "en"
    normalized = language.strip().lower()
    return LANGUAGE_LABEL_TO_CODE.get(normalized, "en")


def format_percent(value: float) -> str:
    return f"{value * 100:.1f}%"


def apply_confidence_threshold(disease: str, confidence: float) -> str:
    if confidence < CONFIDENCE_THRESHOLD:
        return "Uncertain"
    return disease


In [ ]:
%%writefile /kaggle/working/app/predict.py
from io import BytesIO
import warnings
import numpy as np
from PIL import Image, ImageOps
from .model import get_model, model_metadata
from .utils import apply_confidence_threshold

MAX_UPLOAD_BYTES = 10 * 1024 * 1024
MAX_PIXELS = 20_000_000
ALLOWED_CONTENT_TYPES = {"image/jpeg", "image/png", "image/jpg", "image/webp"}
CLASS_NAMES = ["No Finding", "Pneumonia"]

def validate_upload(content_type):
    if not content_type or content_type.lower() not in ALLOWED_CONTENT_TYPES:
        raise ValueError("invalid_image")

def _load_image(image_bytes):
    if not image_bytes or len(image_bytes) > MAX_UPLOAD_BYTES:
        raise ValueError("invalid_image")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("error", Image.DecompressionBombWarning)
            with Image.open(BytesIO(image_bytes)) as image:
                if image.format not in {"JPEG", "PNG", "WEBP"} or image.width * image.height > MAX_PIXELS:
                    raise ValueError("invalid_image")
                image.verify()
            with Image.open(BytesIO(image_bytes)) as image:
                return ImageOps.exif_transpose(image).convert("RGB")
    except Exception as exc:
        raise ValueError("invalid_image") from exc

def preprocess_image(image_bytes):
    image = _load_image(image_bytes).resize((224, 224), Image.Resampling.BILINEAR)
    pixels = np.asarray(image, dtype=np.float32) / 255
    pixels = (pixels - np.array([.485, .456, .406], dtype=np.float32)) / np.array([.229, .224, .225], dtype=np.float32)
    return np.ascontiguousarray(pixels.transpose(2, 0, 1)[None])

def predict_from_image_bytes(image_bytes):
    tensor = preprocess_image(image_bytes)
    session = get_model()
    logits = np.asarray(session.run(None, {session.get_inputs()[0].name: tensor})[0][0], dtype=np.float64)
    if logits.shape != (2,) or not np.isfinite(logits).all():
        raise RuntimeError("Invalid model output")
    logits /= float(model_metadata()["temperature"])
    probabilities = np.exp(logits - logits.max())
    probabilities /= probabilities.sum()
    index = int(probabilities.argmax())
    confidence = float(probabilities[index])
    return apply_confidence_threshold(CLASS_NAMES[index], confidence), confidence


In [ ]:
%%writefile /kaggle/working/clean_nih.py
"""Clean NIH images without modifying raw files; emit leakage-safe manifests and EDA."""
import argparse
from collections import Counter
import hashlib
import html
import json
from pathlib import Path
import warnings

import imagehash
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, n):
        while self.parent[n] != n:
            self.parent[n] = self.parent[self.parent[n]]
            n = self.parent[n]
        return n
    def join(self, a, b): self.parent[self.find(a)] = self.find(b)

class HashTree:
    """BK-tree for bounded Hamming-distance lookup (avoids all-pairs image checks)."""
    def __init__(self): self.root = None
    def add(self, value, index):
        if self.root is None:
            self.root = [value, index, {}]; return
        node = self.root
        while True:
            distance = (value ^ node[0]).bit_count()
            if distance not in node[2]:
                node[2][distance] = [value, index, {}]; return
            node = node[2][distance]
    def nearby(self, value, tolerance=2):
        stack = [self.root] if self.root else []
        while stack:
            node = stack.pop()
            distance = (value ^ node[0]).bit_count()
            if distance <= tolerance: yield node[1]
            stack.extend(child for edge, child in node[2].items() if distance-tolerance <= edge <= distance+tolerance)

def clean(raw, out, seed=42):
    raw, out = Path(raw), Path(out)
    out.mkdir(parents=True, exist_ok=True)
    metadata_path = raw / "Data_Entry_2017_v2020.csv"
    if not metadata_path.is_file():
        metadata_path = raw / "Data_Entry_2017.csv"
    metadata = pd.read_csv(metadata_path)
    metadata = metadata.rename(columns={"Patient Sex": "Patient Gender"})
    required = {"Image Index", "Finding Labels", "Patient ID", "Patient Age", "Patient Gender", "View Position"}
    if not required.issubset(metadata.columns):
        raise ValueError("Missing NIH metadata columns")
    test_names = set((raw / "test_list.txt").read_text().splitlines())
    train_names = set((raw / "train_val_list.txt").read_text().splitlines())
    if test_names & train_names: raise ValueError("Official lists overlap")
    test_patients = set(metadata.loc[metadata["Image Index"].isin(test_names), "Patient ID"].astype(str))
    paths = {}
    duplicate_paths = set()
    for path in raw.rglob("*.png"):
        if path.name in paths: duplicate_paths.add(path.name)
        paths[path.name] = path
    rows, excluded = [], []
    completeness = metadata.isna().sum().to_dict()
    def reject(name, reason): excluded.append({"image": str(name), "reason":reason})
    seen_metadata = set()
    duplicate_metadata = set(metadata.loc[metadata["Image Index"].duplicated(keep=False), "Image Index"])
    for record in metadata.to_dict("records"):
        name = record["Image Index"]
        if name in duplicate_metadata:
            reject(name, "duplicate_metadata"); continue
        if name in seen_metadata:
            reject(name, "duplicate_metadata"); continue
        seen_metadata.add(name)
        if name in duplicate_paths:
            reject(name, "ambiguous_image_path"); continue
        if name not in paths:
            reject(name, "missing_image"); continue
        labels = str(record["Finding Labels"])
        patient = record["Patient ID"]
        if pd.isna(patient) or pd.isna(record["Finding Labels"]) or not str(patient).isdigit():
            reject(name, "invalid_metadata"); continue
        label = 1 if "Pneumonia" in labels.split("|") else 0 if labels == "No Finding" else None
        if label is None:
            reject(name, "outside_binary_scope"); continue
        if name not in test_names and name not in train_names:
            reject(name, "missing_official_split"); continue
        path = paths[name]
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("error", Image.DecompressionBombWarning)
                with Image.open(path) as image:
                    image.verify()
                with Image.open(path) as image:
                    if image.width * image.height > 20_000_000: raise ValueError("too many pixels")
                    gray = image.convert("L")
                    brightness = float(np.asarray(gray.resize((64, 64))).mean())
                    phash = int(str(imagehash.phash(gray)), 16)
                    width, height, mode = image.width, image.height, image.mode
            digest = hashlib.sha256(path.read_bytes()).hexdigest()
        except Exception:
            reject(name, "corrupt_or_oversized_image"); continue
        rows.append({"image":name, "path":str(path.resolve()), "patient":str(patient), "label":label,
            "labels":labels, "official_split":"test" if name in test_names else "development",
            "width":width, "height":height, "mode":mode, "brightness":brightness, "sha256":digest,
            "phash":phash, "age":record["Patient Age"], "sex":record["Patient Gender"], "view":record["View Position"],
            "review":brightness < 5 or brightness > 250 or width < 224 or height < 224
                or pd.isna(record["Patient Age"]) or not 0 <= float(record["Patient Age"]) <= 120})
    uf, tree = UnionFind(len(rows)), HashTree()
    exact = {}
    for index, row in enumerate(rows):
        if row["sha256"] in exact: uf.join(index, exact[row["sha256"]])
        exact[row["sha256"]] = index
        for match in tree.nearby(row["phash"]): uf.join(index, match)
        tree.add(row["phash"], index)
    groups = {}
    for index in range(len(rows)): groups.setdefault(uf.find(index), []).append(index)
    kept = []
    for members in groups.values():
        group_rows = [rows[i] for i in members]
        if len({r["label"] for r in group_rows}) > 1:
            for row in group_rows: reject(row["image"], "conflicting_duplicate_labels")
            continue
        # Keep test preferentially. Remove all development members of cross-split duplicate groups.
        eligible = [r for r in group_rows if r["official_split"] == "test"]
        if not eligible: eligible = group_rows
        selected = sorted(eligible, key=lambda r:r["image"])[0]
        selected["duplicate_group"] = selected["image"]
        for row in group_rows:
            if row is not selected: reject(row["image"], "exact_or_near_duplicate")
        if selected["official_split"] != "test" and selected["patient"] in test_patients:
            reject(selected["image"], "patient_in_official_test"); continue
        kept.append(selected)
    if not kept: raise ValueError("No eligible decoded images; inspect download and exclusion log")
    frame = pd.DataFrame(kept)
    development = frame[frame.official_split == "development"]
    patient_labels = development.groupby("patient").label.max()
    if len(patient_labels) < 4 or patient_labels.value_counts().min() < 2 or len(patient_labels.unique()) != 2:
        raise ValueError("Insufficient development patients per class for a stratified split")
    _, val_patients = train_test_split(patient_labels.index, test_size=.2, random_state=seed, stratify=patient_labels.values)
    frame["split"] = np.where(frame.official_split == "test", "test", np.where(frame.patient.isin(val_patients), "val", "train"))
    assert frame.groupby("patient").split.nunique().max() == 1
    for split in ["train", "val", "test"]:
        if set(frame.loc[frame.split == split, "label"]) != {0, 1}:
            raise ValueError(f"{split} needs both classes; obtain more official archives")
    frame.to_csv(out / "manifest.csv", index=False)
    pd.DataFrame(excluded, columns=["image","reason"]).to_csv(out / "exclusions.csv", index=False)
    info = {"source":"https://nihcc.app.box.com/v/ChestXray-NIHCC", "seed":seed,
        "metadata_sha256":hashlib.sha256(metadata_path.read_bytes()).hexdigest(),
        "manifest_sha256":hashlib.sha256((out / "manifest.csv").read_bytes()).hexdigest(),
        "metadata_rows":len(metadata), "images_found":len(paths), "clean_rows":len(frame),
        "clean_patients":int(frame.patient.nunique()),
        "metadata_label_counts":metadata["Finding Labels"].str.split("|").explode().value_counts().to_dict(),
        "scope":"complete" if len(paths) == len(metadata) else "partial",
        "excluded":dict(Counter(r["reason"] for r in excluded)), "missing_metadata":completeness,
        "split_counts":{f"{split}:{label}":int(count) for (split,label),count in frame.groupby(["split","label"]).size().items()},
        "review_flags":int(frame.review.sum()), "label_limitation":"Report-derived labels; No Finding does not establish health."}
    (out / "cleaning.json").write_text(json.dumps(info, indent=2, default=int), encoding="utf-8")
    report(frame, info, out)
    return frame, info

def report(frame, info, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    figures = []
    plots = {
        "classes":lambda ax: frame.assign(label=frame.label.map({0:"No Finding",1:"Pneumonia"})).groupby(["split","label"]).size().unstack(fill_value=0).plot.bar(ax=ax),
        "dimensions":lambda ax: ax.scatter(frame.width, frame.height, s=4, alpha=.3),
        "brightness":lambda ax: frame.brightness.hist(ax=ax, bins=30),
        "ages":lambda ax: pd.to_numeric(frame.age, errors="coerce").hist(ax=ax, bins=30),
        "views":lambda ax: frame.view.value_counts().plot.bar(ax=ax),
    }
    for name, draw in plots.items():
        fig, ax = plt.subplots(figsize=(7,4)); draw(ax); ax.set_title(name)
        if name == "dimensions": ax.set_xlabel("Width (pixels)"); ax.set_ylabel("Height (pixels)")
        elif name == "brightness": ax.set_xlabel("Mean grayscale intensity (0–255)"); ax.set_ylabel("Images")
        elif name == "ages": ax.set_xlabel("Patient age (years)"); ax.set_ylabel("Images")
        else: ax.set_ylabel("Images")
        fig.tight_layout(); fig.savefig(out / f"{name}.png"); plt.close(fig)
        figures.append(f'<h2>{name}</h2><img src="{name}.png" alt="{name} chart" width="700">')
    fig, axes = plt.subplots(2, 4, figsize=(12,6))
    for label in [0,1]:
        samples = frame[frame.label == label].head(4)
        for ax, row in zip(axes[label], samples.to_dict("records")):
            with Image.open(row["path"]) as image: ax.imshow(image, cmap="gray")
            ax.set_title("Pneumonia" if label else "No Finding"); ax.axis("off")
    fig.tight_layout(); fig.savefig(out / "samples.png"); plt.close(fig)
    document = '<!doctype html><html lang="en"><meta charset="utf-8"><title>NIH cleaning and EDA</title><body><h1>NIH ChestX-ray14 cleaning and EDA</h1><p>Report-derived labels. This report describes the downloaded subset only. Unusual images are flagged, not removed solely for brightness.</p><pre>'
    document += html.escape(json.dumps(info, indent=2, default=int)) + '</pre>' + ''.join(figures) + '<h2>Representative samples</h2><img src="samples.png" width="900"></body></html>'
    (out / "eda.html").write_text(document, encoding="utf-8")
    import nbformat
    notebook = nbformat.v4.new_notebook(cells=[
        nbformat.v4.new_markdown_cell("# NIH ChestX-ray14 EDA\nReport-derived labels; patient-separated splits; raw data stays unchanged."),
        nbformat.v4.new_code_cell("from pathlib import Path\nimport pandas as pd\nfrom IPython.display import display, Image\nroot = Path('.')\ndf = pd.read_csv(root / 'manifest.csv')\ndisplay(df.groupby(['split', 'label']).size().unstack(fill_value=0))\ndisplay(df[['width','height','brightness','age']].describe())"),
        nbformat.v4.new_code_cell("for name in ['classes','dimensions','brightness','ages','views','samples']:\n    display(Image(filename=str(root / (name + '.png'))))")
    ])
    nbformat.write(notebook, out / "eda.ipynb")

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--raw", type=Path, default=Path("data/raw"))
    parser.add_argument("--out", type=Path, default=Path("data/processed"))
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()
    _, info = clean(args.raw, args.out, args.seed)
    print(json.dumps(info, indent=2, default=int))


In [ ]:
%%writefile /kaggle/working/train.py
"""Manifest-driven transfer learning, validation selection, calibration and ONNX export."""
import argparse
import copy
import hashlib
import json
import random
from pathlib import Path
import time

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, brier_score_loss)
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

CLASSES = ["No Finding", "Pneumonia"]

class ManifestImages(Dataset):
    def __init__(self, frame, training=False):
        self.rows = frame.to_dict("records")
        # Identical resize implementation to deployed numpy/Pillow preprocessing.
        self.transform = transforms.Compose(
            ([transforms.RandomRotation(7)] if training else []) +
            [transforms.ToTensor(), transforms.Normalize([.485,.456,.406],[.229,.224,.225])])
    def __len__(self): return len(self.rows)
    def __getitem__(self, index):
        row = self.rows[index]
        with Image.open(row["path"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB").resize((224,224), Image.Resampling.BILINEAR)
            tensor = self.transform(image)
        return tensor, int(row["label"])

def build_model(pretrained=False):
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model

def probabilities(logits, temperature=1):
    scaled = logits / temperature
    scaled -= scaled.max(axis=1, keepdims=True)
    values = np.exp(scaled)
    return values[:,1] / values.sum(axis=1)

def metrics(labels, logits, temperature=1):
    scores = probabilities(logits, temperature)
    predicted = scores >= .5
    tn, fp, fn, tp = confusion_matrix(labels, predicted, labels=[0,1]).ravel()
    ece = 0.
    for low in np.linspace(0, .9, 10):
        members = (scores >= low) & (scores < low+.1 if low < .9 else scores <= 1)
        if members.any(): ece += members.mean() * abs(scores[members].mean() - labels[members].mean())
    return {"sensitivity":float(recall_score(labels,predicted,zero_division=0)),
        "specificity":float(tn / max(tn+fp,1)), "precision":float(precision_score(labels,predicted,zero_division=0)),
        "f1":float(f1_score(labels,predicted,zero_division=0)), "balanced_accuracy":float(balanced_accuracy_score(labels,predicted)),
        "roc_auc":float(roc_auc_score(labels,scores)) if len(np.unique(labels)) == 2 else None,
        "brier":float(brier_score_loss(labels,scores)), "ece":float(ece),
        "confusion_matrix":[[int(tn),int(fp)],[int(fn),int(tp)]],
        "abstention_fraction":float((np.maximum(scores,1-scores) < .8).mean())}

def collect(model, loader, device):
    outputs, labels = [], []
    model.eval()
    with torch.inference_mode():
        for images, targets in loader:
            outputs.append(model(images.to(device)).cpu().numpy()); labels.append(targets.numpy())
    return np.concatenate(labels), np.concatenate(outputs)

def calibrate(logits, labels):
    # Bounded temperature chosen by validation NLL only.
    candidates = np.geomspace(.05,20,200)
    losses = []
    for temperature in candidates:
        scores = np.clip(probabilities(logits.copy(),temperature),1e-7,1-1e-7)
        losses.append(float(-(labels*np.log(scores)+(1-labels)*np.log(1-scores)).mean()))
    return float(candidates[int(np.argmin(losses))])

def validate_manifest(frame):
    required = {"path","label","patient","split","sha256","duplicate_group"}
    if not required.issubset(frame.columns): raise ValueError("Run clean_nih.py first")
    for field in ["patient","sha256","duplicate_group"]:
        if frame.groupby(field).split.nunique().max() > 1: raise ValueError(f"{field} leakage")
    for split in ["train","val","test"]:
        if set(frame.loc[frame.split == split,"label"]) != {0,1}: raise ValueError(f"{split} lacks both classes")

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--manifest", type=Path, default=Path("data/processed/manifest.csv"))
    parser.add_argument("--out", type=Path, default=Path("artifacts"))
    parser.add_argument("--baseline", type=Path, default=Path("models/pneumonia_model.pth"))
    parser.add_argument("--epochs", type=int, default=10)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--evaluate-only", action="store_true")
    args = parser.parse_args()
    if args.epochs < 1 or args.batch_size < 1: parser.error("epochs and batch-size must be positive")
    torch.manual_seed(args.seed); random.seed(args.seed); np.random.seed(args.seed)
    torch.set_num_threads(2)
    torch.use_deterministic_algorithms(True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    frame = pd.read_csv(args.manifest)
    validate_manifest(frame)
    args.out.mkdir(parents=True, exist_ok=True)
    loaders = {split:DataLoader(ManifestImages(frame[frame.split==split], split=="train"),
        batch_size=args.batch_size,shuffle=split=="train",num_workers=4,pin_memory=device.type=="cuda") for split in ["train","val","test"]}
    report = {"seed":args.seed,"device":str(device),
        "manifest_sha256":hashlib.sha256(args.manifest.read_bytes()).hexdigest(),
        "baseline_provenance":"Unknown; historic training overlap cannot be excluded", "epochs":[]}
    baseline, baseline_metrics = None, None
    try:
        baseline = build_model().to(device)
        state = torch.load(args.baseline, map_location=device, weights_only=True)
        baseline.load_state_dict(state)
        labels, logits = collect(baseline,loaders["val"],device)
        baseline_metrics = metrics(labels,logits)
        report["baseline_validation"] = baseline_metrics
    except Exception as exc:
        report["baseline_error"] = f"{type(exc).__name__}: {exc}"
        baseline = None
    if args.evaluate_only:
        if baseline is not None:
            labels, logits = collect(baseline,loaders["test"],device)
            report["baseline_test"] = metrics(labels,logits)
        (args.out/"evaluation.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
        return
    candidate = build_model(pretrained=True).to(device)
    for name, parameter in candidate.named_parameters():
        parameter.requires_grad = name.startswith(("layer4.","fc."))
    counts = frame.loc[frame.split=="train","label"].value_counts()
    weights = torch.tensor([1/counts[0],1/counts[1]],dtype=torch.float32,device=device)
    criterion = nn.CrossEntropyLoss(weight=weights/weights.mean())
    optimizer = torch.optim.AdamW(filter(lambda p:p.requires_grad,candidate.parameters()), lr=1e-4,weight_decay=1e-4)
    best, best_score, stale = None, -1., 0
    for epoch in range(args.epochs):
        candidate.eval() # Frozen backbone batchnorm statistics stay fixed.
        candidate.layer4.train(); candidate.fc.train()
        for images, labels in loaders["train"]:
            optimizer.zero_grad()
            loss = criterion(candidate(images.to(device)),labels.to(device))
            loss.backward(); optimizer.step()
        labels, logits = collect(candidate,loaders["val"],device)
        measured = metrics(labels,logits)
        report["epochs"].append({"epoch":epoch+1,**measured})
        print(json.dumps(report["epochs"][-1]),flush=True)
        if measured["balanced_accuracy"] > best_score:
            best, best_score, stale = copy.deepcopy(candidate.state_dict()), measured["balanced_accuracy"], 0
        else: stale += 1
        if stale >= 3: break
    candidate.load_state_dict(best)
    # Preserve the best research candidate even if it fails the promotion gate.
    torch.save(candidate.state_dict(),args.out/"candidate-research.pth")
    labels, logits = collect(candidate,loaders["val"],device)
    candidate_metrics = metrics(labels,logits)
    promote = baseline_metrics is None or (
        candidate_metrics["balanced_accuracy"] > baseline_metrics["balanced_accuracy"] and
        candidate_metrics["sensitivity"] >= baseline_metrics["sensitivity"])
    # Model selection is frozen here, before consulting test data.
    selected = candidate if promote else baseline
    labels, logits = collect(selected,loaders["val"],device)
    temperature = calibrate(logits,labels)
    report.update({"candidate_validation":candidate_metrics, "selected":"candidate" if promote else "baseline",
        "temperature":temperature,"calibrated_validation":metrics(labels,logits,temperature)})
    labels, logits = collect(selected,loaders["test"],device)
    report["selected_test"] = metrics(labels,logits,temperature)
    if not promote:
        report["deployment_blocked"] = "Legacy baseline retained for evaluation only: unknown provenance and historic Normal class semantics"
        (args.out/"evaluation.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
        return
    selected.cpu().eval()
    torch.save(selected.state_dict(),args.out/"candidate.pth")
    onnx_path = args.out/"pneumonia_model.onnx"
    torch.onnx.export(selected,torch.zeros(1,3,224,224),str(onnx_path),input_names=["image"],output_names=["logits"],
                      opset_version=17,dynamo=False)
    import onnxruntime as ort
    session = ort.InferenceSession(str(onnx_path),providers=["CPUExecutionProvider"])
    max_error = 0.
    for row in frame[frame.split=="val"].head(8).to_dict("records"):
        from app.predict import preprocess_image
        array = preprocess_image(Path(row["path"]).read_bytes())
        with torch.inference_mode(): expected = selected(torch.from_numpy(array)).numpy()
        actual = session.run(None,{"image":array})[0]
        max_error = max(max_error,float(np.max(np.abs(actual-expected))))
    if max_error > 1e-4: raise RuntimeError(f"ONNX parity failed: {max_error}")
    report["onnx_max_absolute_error"] = max_error
    version = "nih-resnet18-" + report["manifest_sha256"][:12]
    metadata = {"validated":True,"version":version,"classes":CLASSES,"temperature":temperature,
        "calibrated":True,"sha256":hashlib.sha256(onnx_path.read_bytes()).hexdigest(),
        "manifest_sha256":report["manifest_sha256"],"test_metrics":report["selected_test"],
        "limitations":"NIH report-derived labels. No Finding is not confirmed healthy. Educational screening only."}
    onnx_path.with_suffix(".json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
    (args.out/"evaluation.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    print(f"Validated artifacts exported to {args.out}; run benchmark_inference.py before deployment.")

if __name__ == "__main__": main()


In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU count:', torch.cuda.device_count())
    assert torch.cuda.device_count() > 0

In [ ]:
import subprocess
clean_cmd = [sys.executable, '/kaggle/working/clean_nih.py', '--raw', str(RAW), '--out', '/kaggle/working/processed', '--seed', '42']
print('Running reproducible NIH cleaning + EDA...')
subprocess.run(clean_cmd, check=True)

In [ ]:
import json
cleaning = json.loads(Path('/kaggle/working/processed/cleaning.json').read_text())
print(json.dumps({k: cleaning[k] for k in ['metadata_rows','images_found','clean_rows','clean_patients','scope','split_counts','excluded','review_flags']}, indent=2))

In [ ]:
train_cmd = [sys.executable, '/kaggle/working/train.py', '--manifest', '/kaggle/working/processed/manifest.csv', '--out', '/kaggle/working/artifacts', '--baseline', '/kaggle/working/pneumonia_model.pth', '--epochs', '10', '--batch-size', '32', '--seed', '42']
print('Training on GPU with early stopping; selected test metrics are evaluated after validation selection.')
subprocess.run(train_cmd, check=True)

In [ ]:
report_path = Path('/kaggle/working/artifacts/evaluation.json')
if report_path.exists():
    print(report_path.read_text())
else:
    print('No evaluation report was produced; candidate did not pass promotion or training failed.')

## Export and deployment
If training exports both pneumonia_model.onnx and its matching JSON metadata, inspect evaluation.json, confirm the manifest/split counts, and download all three artifacts from the artifacts output folder. The backend requires the ONNX file and JSON sidecar together; it verifies the checksum and validated flag at startup. Never deploy an artifact if the validation promotion gate failed.